**New benchmark analysis (2026-10-05).** Uses the available 4L/2H/K16 checkpoints with weight decay **0.1**, seeds 100–104. This is a new exploratory analysis, not reproduction of SEL-4L wd0.01. References/conflict definitions are frozen historical inputs, verified by hash. All model-dependent outputs are recomputed. Validation match splits support the stated comparisons; prior test access prevents a fresh confirmatory claim. Population and intervention scope apply to each reported result. See `config/analysis_plan.json` and `SURGICAL_FIXES.md`.

# MB02 — Checking the Frozen Model, Instrumenting It, and Pre-registration

**Track B · Mechanistic interpretability of SEL-4L · Identity × Context project**

| Item | Content |
|---|---|
| **Questions** | (1) Is the model we load exactly SEL-4L? (2) Does our step-by-step version of the model (used to read and change internal values) give exactly the same outputs? (3) Which groups of events, controls and tests are fixed *before* we look at any result? |
| **Inputs** | MB01 outputs (checked by fingerprint); `track_a/experiments/hyperparameter_optimisation/results/13_final_runs.csv` and the five run folders it lists; `ablations/results/16_validation_report.json` (for the context-only model B5) |
| **Outputs** | `results/MB02_verification.csv`, `results/MB02_instrumentation_tests.csv`, `artifacts/populations/*.parquet`, `artifacts/populations/reference_means.pt`, `config/mi_config.json`, `config/preregistration.json` |
| **Test data** | Not used |
| **Code files written** | `src/mi_setup.py`, `src/hooks.py`, `src/interventions.py`, `src/mi_metrics.py`, `src/attribution.py`, `tests/test_hooks.py`, `tests/test_interventions.py` |

**SEL-4L** is the model under study: a Transformer with 4 layers, 2 attention heads, d_model 128, d_ff 512, a window of K = 16 events, trained with five seeds (100–104).

In [1]:
import os
import sys
import json
import time
import importlib
import importlib.util
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")


def find_root(start):
    """`start` (or one of its ancestors) holding both project folders directly, else None."""
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "model_training_evaluation").is_dir() and (candidate / "mi_analysis").is_dir():
            return candidate
    return None


def search_root(bases, max_depth=6):
    """Breadth-first search under each base (e.g. a mounted Drive) for the project root."""
    for base in bases:
        base = Path(base)
        if not base.is_dir():
            continue
        found = find_root(base)
        if found is not None:
            return found
        frontier = [(base, 0)]
        while frontier:
            folder, depth = frontier.pop(0)
            if depth >= max_depth:
                continue
            try:
                children = [c for c in folder.iterdir() if c.is_dir() and not c.name.startswith(".")]
            except PermissionError:
                continue
            for child in children:
                if child.name == "model_training_evaluation" and (child.parent / "mi_analysis").is_dir():
                    return child.parent
                frontier.append((child, depth + 1))
    return None


if "SOCCER_MI_ROOT" in os.environ:
    ROOT = Path(os.environ["SOCCER_MI_ROOT"])
elif IN_COLAB:
    ROOT = search_root(["/content", "/content/drive/MyDrive", "/content/drive/My Drive"])
    if ROOT is None:
        raise RuntimeError(
            "BLOCKED: could not find a folder holding both model_training_evaluation/ and mi_analysis/ "
            "under /content or the mounted Drive. Upload or sync the project there, or set "
            "SOCCER_MI_ROOT to its exact path in a cell ABOVE this one, for example: "
            "import os; os.environ['SOCCER_MI_ROOT'] = '/content/drive/MyDrive/.../soccer_mi'")
else:
    ROOT = find_root(Path.cwd())
    if ROOT is None:
        raise RuntimeError(f"BLOCKED: no ancestor of {Path.cwd()} holds both model_training_evaluation/ "
                           "and mi_analysis/. Run this notebook from inside the project, or set SOCCER_MI_ROOT.")
TRACK_A_DIR = ROOT / "model_training_evaluation"
MI_DIR = Path(os.environ.get("MI_ANALYSIS_DIR", ROOT / "mi_analysis")).resolve()
assert (TRACK_A_DIR / "src" / "data.py").exists(), (
    f"BLOCKED: {TRACK_A_DIR / 'src' / 'data.py'} not found -- ROOT resolved to {ROOT}, which is wrong. "
    "Set SOCCER_MI_ROOT explicitly.")
assert (TRACK_A_DIR / "artifacts" / "events_prepared.parquet").exists(), (
    f"BLOCKED: {TRACK_A_DIR / 'artifacts' / 'events_prepared.parquet'} not found -- ROOT is wrong.")
RUN_MODE = os.environ.get("MI_RUN_MODE", "full")
assert RUN_MODE in ("full", "smoke")

for folder in [MI_DIR, MI_DIR / "src", MI_DIR / "notebooks", MI_DIR / "tests"]:
    folder.mkdir(parents=True, exist_ok=True)
(MI_DIR / "src" / "__init__.py").touch()
os.chdir(MI_DIR)

sys.path[:] = [entry for entry in sys.path if entry not in ("", ".", str(MI_DIR))]
if str(TRACK_A_DIR) not in sys.path:
    sys.path.insert(0, str(TRACK_A_DIR))
spec = importlib.util.spec_from_file_location("mi_src", MI_DIR / "src" / "__init__.py",
                                              submodule_search_locations=[str(MI_DIR / "src")])
mi_src = importlib.util.module_from_spec(spec)
sys.modules["mi_src"] = mi_src
spec.loader.exec_module(mi_src)

print("Colab:", IN_COLAB, "| RUN_MODE:", RUN_MODE)
print("ROOT:", ROOT)
print("Track A (read only):", TRACK_A_DIR, "| exists:", TRACK_A_DIR.exists())
print("Track B:", MI_DIR)

Colab: False | RUN_MODE: full
ROOT: /home/aayush/code/soccer_mi
Track A (read only): /home/aayush/code/soccer_mi/model_training_evaluation | exists: True
Track B: /home/aayush/code/soccer_mi/.mi_analysis_rerun


## 0. Code files written by this notebook

* `mi_setup.py` — loads SEL-4L exactly as it was saved (using the model-training project's own `build_model` and `load_trained_model`), builds the K = 16 evaluation windows, and finds the window and position where each event is scored.
* `hooks.py` — a **step-by-step forward pass**: the model's computation written out with the model's own layers, with named *sites* where a value inside the model can be recorded or replaced. We use this instead of PyTorch hooks so that every intermediate value is visible in the code. Test 1 below checks it gives exactly the model's outputs.
* `interventions.py` — changes to the model's inputs (remove or swap a player, team or role at one position), each checking that it changed nothing else.
* `mi_metrics.py` — Δ, KL divergence, match-level bootstrap intervals, pooled recovery, and the Benjamini–Hochberg correction for many tests.
* `attribution.py` — splits Δ into one part per model component, with the final LayerNorm held fixed (used by test 6 and by MB09).

In [2]:
# Shared source is versioned with this notebook; do not overwrite repaired dependencies.
assert (MI_DIR / 'src/mi_setup.py').exists(), "Missing dependency: src/mi_setup.py"

In [3]:
# Shared source is versioned with this notebook; do not overwrite repaired dependencies.
assert (MI_DIR / 'src/hooks.py').exists(), "Missing dependency: src/hooks.py"

In [4]:
# Shared source is versioned with this notebook; do not overwrite repaired dependencies.
assert (MI_DIR / 'src/interventions.py').exists(), "Missing dependency: src/interventions.py"

> **⚠ REPAIR NOTE (2026-10-04) — SHARED SOURCE UPDATED — F10.** The `%%writefile` cell below now writes the repaired `mi_metrics.py`. Existing functions are byte-for-byte unchanged; three functions were added (`probability_calibration`, `aggregate_log_odds`, `reliability_bins`) for the corrected MB03 calibration analysis. No MB02 result changes. See `REPAIR_REPORT.md`.

In [5]:
# Shared source is versioned with this notebook; do not overwrite repaired dependencies.
assert (MI_DIR / 'src/mi_metrics.py').exists(), "Missing dependency: src/mi_metrics.py"

In [6]:
# Shared source is versioned with this notebook; do not overwrite repaired dependencies.
assert (MI_DIR / 'src/attribution.py').exists(), "Missing dependency: src/attribution.py"

In [7]:
# Shared source is versioned with this notebook; do not overwrite repaired dependencies.
assert (MI_DIR / 'tests/test_hooks.py').exists(), "Missing dependency: tests/test_hooks.py"

In [8]:
# Shared source is versioned with this notebook; do not overwrite repaired dependencies.
assert (MI_DIR / 'tests/test_interventions.py').exists(), "Missing dependency: tests/test_interventions.py"

In [9]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import src.data as data
import src.metrics as metrics
import src.training as training
import mi_src.paths as paths
import mi_src.fingerprint as fingerprint
import mi_src.conflict as conflict
import mi_src.mi_setup as mi_setup
import mi_src.hooks as hooks
import mi_src.interventions as interventions
import mi_src.mi_metrics as mi_metrics
import mi_src.attribution as attribution
for module in [data, metrics, training, paths, fingerprint, conflict, mi_setup, hooks, interventions, mi_metrics, attribution]:
    importlib.reload(module)

if getattr(paths, "SCHEMA_VERSION", 1) < 2:
    raise RuntimeError("BLOCKED: mi_analysis/src/paths.py is older than this notebook expects. "
                       "Re-run MB01 section 0 (its three %%writefile cells), then restart the kernel.")
paths.check_modules(**{name: module for name, module in
                       [("paths", paths), ("fingerprint", fingerprint), ("mi_setup", mi_setup),
                        ("patching", globals().get("patching")), ("circuits", globals().get("circuits"))]
                       if module is not None})

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

OUT = paths.output_dir(MI_DIR, RUN_MODE)
SETTINGS = paths.RUN_SETTINGS[RUN_MODE]
TRACK = paths.track_a_paths(TRACK_A_DIR)
MANIFEST = OUT / "manifests" / "sha_manifest.json"
REGISTRY = OUT / "manifests" / "experiment_registry.jsonl"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(int(os.environ.get("MI_CPU_THREADS", "4")))
if RUN_MODE == "full" and DEVICE != "cuda":
    print("Note: no GPU found. The full run works on CPU but is much slower.")
training.configure_determinism()
training.set_seed(0)
ACTIONS = list(data.ACTION_NAMES)
print("Device:", DEVICE, "| outputs:", OUT)
print("Settings:", SETTINGS)

verified = fingerprint.verify_files(MANIFEST)
print(f"\nMB01 outputs verified by SHA-256: {len(verified)} files")

Note: no GPU found. The full run works on CPU but is much slower.


Device: cpu | outputs: /home/aayush/code/soccer_mi/.mi_analysis_rerun
Settings: {'reference_epochs': 25, 'batch_windows': 256, 'n_bootstrap': 2000, 'n_draws': 5, 'n_permutations': 1000, 'n_random_circuits': 100, 'probe_train_events': 20000, 'max_events': 100000, 'probe_seeds': 3, 'scrub_draws': 5}

MB01 outputs verified by SHA-256: 32 files


> **⚠ REPAIR (2026-10-04) — F26: MB02 can now be run again safely.** Before this repair, every run of MB02
> (1) overwrote `manifests/track_a_snapshot.json`, which MB01 wrote and fingerprinted, with a new layout and a timestamp, so the
> next notebook stopped with `BLOCKED: 'track_a_snapshot.json' ... has changed`; (2) rewrote `config/preregistration.json` with a
> new date; and (3) regenerated the population files.
>
> Now every check still runs, but **no file an earlier run wrote is ever overwritten**. Each output is first written to a side file
> and compared by *content* with the existing one (tables row by row, tensors exactly, the preregistration ignoring its
> `written_at` date). Same content → the side file is deleted and the original file, with its original fingerprint, is kept.
> Different content → the original is still kept, the new version stays next to it as `*.rerun_<time>.*`, and the notebook
> stops at the end and lists the differences. MB01's Track A snapshot is only *compared*, never rewritten.
> A first run on a fresh project writes the files exactly as before.

In [10]:
# REPAIR F26 — write-once outputs. See the note above.
from datetime import datetime, timezone

RERUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_LOG = []          # (file, what happened)
RERUN_DIFFERENCES = []   # (file, side file, detail) — only when a re-run produced different content


def _same_object(a, b, tolerance):
    """(equal, largest numeric difference) for nested dicts/lists of tensors, arrays and plain values."""
    if isinstance(a, dict) and isinstance(b, dict):
        if set(a) != set(b):
            return False, float("inf")
        worst = 0.0
        for key in a:
            ok, diff = _same_object(a[key], b[key], tolerance)
            worst = max(worst, diff)
            if not ok:
                return False, worst
        return True, worst
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        if len(a) != len(b):
            return False, float("inf")
        worst = 0.0
        for x, y in zip(a, b):
            ok, diff = _same_object(x, y, tolerance)
            worst = max(worst, diff)
            if not ok:
                return False, worst
        return True, worst
    if torch.is_tensor(a) or torch.is_tensor(b):
        a, b = torch.as_tensor(a).cpu(), torch.as_tensor(b).cpu()
        if a.shape != b.shape:
            return False, float("inf")
        diff = float((a.double() - b.double()).abs().max()) if a.numel() else 0.0
        return diff <= tolerance, diff
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        a, b = np.asarray(a), np.asarray(b)
        if a.shape != b.shape:
            return False, float("inf")
        if a.dtype.kind in "fc" or b.dtype.kind in "fc":
            diff = float(np.abs(a.astype(float) - b.astype(float)).max()) if a.size else 0.0
            return diff <= tolerance, diff
        return bool(np.array_equal(a, b)), 0.0
    return a == b, 0.0


def same_content(old_path, new_path, tolerance=0.0):
    """(equal, detail). Tables row by row, tensors to `tolerance`, JSON ignoring its written_at date."""
    suffix = Path(old_path).suffix
    if suffix == ".parquet":
        old, new = pd.read_parquet(old_path), pd.read_parquet(new_path)
        return old.equals(new), f"{len(old):,} rows before, {len(new):,} rows now"
    if suffix == ".csv":
        old, new = pd.read_csv(old_path), pd.read_csv(new_path)
        return old.equals(new), f"{len(old):,} rows before, {len(new):,} rows now"
    if suffix == ".json":
        old, new = json.load(open(old_path)), json.load(open(new_path))
        old.pop("written_at", None); new.pop("written_at", None)
        changed = sorted(k for k in set(old) | set(new) if old.get(k) != new.get(k))
        return not changed, ("keys that differ: " + ", ".join(changed)) if changed else "same apart from written_at"
    if suffix == ".pt":
        ok, diff = _same_object(torch.load(old_path, weights_only=False), torch.load(new_path, weights_only=False), tolerance)
        return ok, f"largest difference {diff:.3g} (tolerance {tolerance:g})"
    return False, "no content comparison for this file type"


def keep_or_write(path, writer, tolerance=0.0):
    """Write `path` if it does not exist. Otherwise never overwrite it: write a side file, compare, keep the original."""
    path = Path(path)
    if not path.exists():
        writer(path)
        OUTPUT_LOG.append((str(path.relative_to(OUT)), "written (first run)"))
        return
    side = path.with_name(f"{path.stem}.rerun_{RERUN_STAMP}{path.suffix}")
    writer(side)
    if fingerprint.file_sha256(side) == fingerprint.file_sha256(path):
        equal, detail = True, "byte-identical"
    else:
        equal, detail = same_content(path, side, tolerance)
    if equal:
        side.unlink()
        OUTPUT_LOG.append((str(path.relative_to(OUT)), f"same content, existing file kept ({detail})"))
    else:
        RERUN_DIFFERENCES.append((str(path.relative_to(OUT)), side.name, detail))
        OUTPUT_LOG.append((str(path.relative_to(OUT)), f"DIFFERENT, existing file kept, new version in {side.name} ({detail})"))


PRIOR_RUN = (OUT / "config" / "preregistration.json").exists()
print("MB02 has run before in this folder:", PRIOR_RUN,
      "-> outputs will be compared, not overwritten" if PRIOR_RUN else "-> outputs will be written")

MB02 has run before in this folder: False -> outputs will be written


Historical result prose removed for the new model; use the freshly executed outputs above.


In [11]:
selected = mi_setup.load_json(TRACK["selected_config"])
runs, PRIMARY_SEED = mi_setup.sel4l_runs(TRACK)
SEEDS = [int(seed) for seed in runs["seed"]]
print("Selected configuration:", json.dumps({key: selected[key] for key in ["context_length", "model", "training"]}))
print("Seeds:", SEEDS, "| primary seed:", PRIMARY_SEED)

events = pd.read_parquet(TRACK["events"])
bundle = mi_setup.build_bundle(events, TRACK, DEVICE)
val_view = bundle["views"]["val"]
targets = bundle["targets"]
match_ids_all = bundle["match_ids"]

models = {}
verification_rows = []
for seed in SEEDS:
    run_dir = mi_setup.run_dir_for_seed(runs, seed)
    run_config = mi_setup.load_json(run_dir / "run_config.json")
    problems = mi_setup.config_mismatches(run_config, selected)
    model = mi_setup.load_model(run_dir, bundle)
    count = __import__("src.model", fromlist=["count_parameters"]).count_parameters(model)["total"]
    models[seed] = model
    verification_rows.append({"seed": seed, "primary": seed == PRIMARY_SEED, "config matches": not problems,
                              "strict load": True, "parameters": count,
                              "best.pt sha256": fingerprint.file_sha256(run_dir / "best.pt")[:16]})
    if problems or count != 911119:
        raise RuntimeError(f"BLOCKED: seed {seed}: {problems or 'parameter count ' + str(count)}")
print(pd.DataFrame(verification_rows).to_string(index=False))
src_hashes = {path.name: fingerprint.file_sha256(path) for path in sorted(TRACK["src"].glob("*.py"))}
print("\nmodel_training_evaluation src/ fingerprints:", {name: digest[:12] for name, digest in src_hashes.items()})

Selected configuration: {"context_length": 16, "model": {"n_layers": 4, "n_heads": 2, "d_model": 128, "d_ff": 512}, "training": {"learning_rate": 0.0003, "weight_decay": 0.1, "max_epochs": 30, "batch_size": 256, "warmup_fraction": 0.05, "min_lr_ratio": 0.1, "grad_clip": 1.0, "patience": 5}}
Seeds: [100, 101, 102, 103, 104] | primary seed: 103


/home/aayush/code/soccer_mi/model_training_evaluation/src/batching.py:19: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  self.tensors[name] = torch.as_tensor(np.asarray(values), device=device)


 seed  primary  config matches  strict load  parameters   best.pt sha256
  100    False            True         True      911119 d9712d0a305481ee
  101    False            True         True      911119 1b04a5520e218025
  102    False            True         True      911119 56134ec75d1d09b3
  103     True            True         True      911119 364652fec05af19d
  104    False            True         True      911119 e0b89ecde7df3a6b

model_training_evaluation src/ fingerprints: {'analysis.py': 'b8075c97883d', 'baselines.py': '0fd32a7c0cd7', 'batching.py': 'b4d6b7691f84', 'data.py': '0b0e0eedf2d5', 'evaluation.py': 'fdf20e7d2b25', 'metrics.py': 'f7cd09c0d62a', 'mlp_baseline.py': '272e1a102aa9', 'model.py': 'c4e79355e13e', 'training.py': '3c1cbc83893b'}


Historical result prose removed for the new model; use the freshly executed outputs above.


In [12]:
RECORDED_PRIMARY_NLL_NB13 = mi_setup.load_json(mi_setup.run_dir_for_seed(runs, PRIMARY_SEED) / "final_results.json")["val_metrics"]["nll"]
reproduction_rows = []
for seed in SEEDS:
    run_dir = mi_setup.run_dir_for_seed(runs, seed)
    logits, val_targets, rows = training.predict_view(models[seed], bundle, "val")
    stored = np.load(run_dir / "val_predictions.npz")
    same_rows = np.array_equal(stored["event_rows"], rows)
    max_difference = float(np.abs(stored["logits"] - logits).max()) if same_rows else np.nan
    nll = metrics.compute_metrics(logits, val_targets)["nll"]
    recorded = mi_setup.load_json(run_dir / "final_results.json")["val_metrics"]["nll"]
    reproduction_rows.append({"seed": seed, "same events": same_rows, "max |logit difference|": max_difference,
                              "recomputed val NLL": nll, "recorded val NLL": recorded,
                              "passed": same_rows and max_difference < 1e-5 and abs(nll - recorded) < 1e-5})
reproduction = pd.DataFrame(reproduction_rows)
# REPAIR F26: write-once (was: reproduction.to_csv(OUT / "results" / "MB02_verification.csv", index=False))
keep_or_write(OUT / "results" / "MB02_verification.csv", lambda p: reproduction.to_csv(p, index=False))
print(reproduction.to_string(index=False))
print(f"\nNotebook 13 recorded val NLL {RECORDED_PRIMARY_NLL_NB13} for the primary seed 103.")
if not reproduction["passed"].all():
    raise RuntimeError("BLOCKED: a checkpoint does not reproduce its recorded validation outputs.")
MODEL = models[PRIMARY_SEED]

 seed  same events  max |logit difference|  recomputed val NLL  recorded val NLL  passed
  100         True                0.000006            0.906943          0.906943    True
  101         True                0.000006            0.902118          0.902118    True
  102         True                0.000008            0.904126          0.904126    True
  103         True                0.000006            0.903776          0.903776    True
  104         True                0.000007            0.898426          0.898426    True

Notebook 13 recorded val NLL 0.9037757515907288 for the primary seed 103.


Historical result prose removed for the new model; use the freshly executed outputs above.


## 4. Tests of the step-by-step forward pass

The step-by-step forward pass is only useful if it computes exactly the same thing as the model. Nine tests check this, and check that input changes only touch what they should. They run on 32 real validation windows.

| # | Test | Passes when |
|---|---|---|
| 1 | No change | the step-by-step logits equal `model(inputs)` exactly |
| 2 | Replace with itself | replacing every site by its own value changes the logits by < 1e-6 |
| 3 | Residual sums | resid_post = resid_pre + attn_out + mlp_out, and the input vector = sum of the factor vectors (< 1e-5) |
| 4 | Head split | the attention output = sum of each head's part + bias (< 1e-5) |
| 5 | No effect backwards in time | a change at position p changes nothing before p (exactly) |
| 6 | Δ split into parts | component parts + constant = Δ (< 1e-4) |
| 7 | Scope of input changes | removing or swapping a player changes only the player at the target; a wrong change is caught |
| 7b | Random vector of the same size | same length (norm) at the target, nothing else changed |
| 8 | Removing a player who has no vector | removing a player already at index 0 changes nothing |
| 9 | CPU and GPU agree | < 1e-4 (skipped without a GPU) |

In [13]:
for name in list(sys.modules):
    if name == "tests" or name.startswith("tests."):
        del sys.modules[name]
(MI_DIR / "tests" / "__init__.py").touch()
if str(MI_DIR) not in sys.path:
    sys.path.insert(0, str(MI_DIR))
import tests.test_hooks as test_hooks
import tests.test_interventions as test_interventions
importlib.reload(test_hooks); importlib.reload(test_interventions)

test_inputs, _ = mi_setup.window_inputs(bundle, val_view, np.arange(32))
test_positions = torch.as_tensor(np.random.default_rng(0).integers(1, 16, 32), device=DEVICE)
context = {"model": MODEL, "inputs": test_inputs, "positions": test_positions}
results = test_hooks.run_all(context) + test_interventions.run_all(context)
test_table = pd.DataFrame(results).sort_values("test")
# REPAIR F26: write-once (was: test_table.to_csv(OUT / "results" / "MB02_instrumentation_tests.csv", index=False))
keep_or_write(OUT / "results" / "MB02_instrumentation_tests.csv", lambda p: test_table.to_csv(p, index=False))
pd.set_option("display.max_colwidth", 120)
print(test_table.to_string(index=False))
if not test_table["passed"].all():
    raise RuntimeError("BLOCKED: an instrumentation test failed.")

                                              test  passed                                                                                detail
1 no-op: instrumented forward equals model(inputs)    True                                       max |difference| = 0.00e+00 (must be exactly 0)
              2 self-patch leaves logits unchanged    True                      replacing each of 51 sites by its own value: max change 0.00e+00
                             3 residual arithmetic    True                                                     max reconstruction error 0.00e+00
              4 head split of the attention output    True              sum of head contributions + bias vs attention output: max error 1.19e-07
                      5 causal locality of a patch    True                                            change at positions < 9: 0.0; at >= 9: 5.3
    6 frozen-LayerNorm decomposition sums to Delta    True                     sum of 25 component terms + constant vs Delta: max 

Historical result prose removed for the new model; use the freshly executed outputs above.


In [14]:
conflict_set = pd.read_parquet(OUT / "artifacts" / "conflict" / "conflict_set.parquet")
val_conflicts = conflict_set[conflict_set.split == "val"].reset_index(drop=True)
windows, positions = mi_setup.locate(val_view, val_conflicts["event_row"].to_numpy())
val_conflicts["window"] = windows
val_conflicts["position"] = positions
print(f"validation conflict rows placed: {len(val_conflicts):,} | scored at position < 8: {(positions < 8).sum()}")

validation conflict rows placed: 10,691 | scored at position < 8: 31


Historical result prose removed for the new model; use the freshly executed outputs above.


In [15]:
features = {name: tensor.cpu().numpy() for name, tensor in bundle["encoded"].tensors.items()}
val_probs = pd.read_parquet(OUT / "artifacts" / "conflict" / "predictions" / "val_probs.parquet")
context_top = pd.Series(np.stack(val_probs["P_ctx"].to_numpy()).argmax(1), index=val_probs["event_row"].to_numpy())

val_conflicts["population"] = np.where(val_conflicts["has_dedicated_player"], "P-player", "index-0")
val_conflicts["genuine"] = val_conflicts["coupled_with_prev"] == 0

val_matches = np.sort(events.loc[events.split == "val", "match_id"].unique())
shuffled_matches = np.random.default_rng(0).permutation(val_matches)
discovery_matches = sorted(int(m) for m in shuffled_matches[:18])
val_conflicts["half"] = np.where(val_conflicts["match_id"].isin(discovery_matches), "discovery", "estimation")

b5_dir = mi_setup.restricted_run_dir(TRACK, "B5 context-only", seed=100)
b5_model = mi_setup.load_model(b5_dir, bundle)
b5_top = np.empty(len(val_conflicts), dtype=np.int64)
with torch.no_grad():
    for part, inputs, pos, a_ctx, a_id in mi_setup.event_batches(bundle, val_view, val_conflicts, SETTINGS["batch_windows"]):
        b5_top[part] = mi_metrics.at_target(b5_model(inputs), pos).argmax(-1).cpu().numpy()
val_conflicts["b5_agrees_with_a_ctx"] = b5_top == val_conflicts["a_ctx"].to_numpy()
del b5_model

id_probabilities = np.stack(val_probs["P_id"].to_numpy())
id_row_of = pd.Series(np.arange(len(val_probs)), index=val_probs["event_row"].to_numpy())
id_rows = id_row_of.loc[val_conflicts["event_row"].to_numpy()].to_numpy()
ranked_id = np.sort(id_probabilities[id_rows], axis=1)
val_conflicts["a_id_margin"] = ranked_id[:, -1] - ranked_id[:, -2]
val_conflicts["a_id_margin_band"] = pd.cut(val_conflicts["a_id_margin"], [-1.0, 0.02, 0.05, 1.0],
                                           labels=["< 0.02", "0.02 to 0.05", "> 0.05"]).astype(str)

if RUN_MODE == "smoke":
    parts = []
    for population_name, group in val_conflicts.groupby("population"):
        parts.append(group.sample(min(len(group), SETTINGS["max_events"]), random_state=0))
    val_conflicts = pd.concat(parts).sort_values("event_row").reset_index(drop=True)
print(val_conflicts.groupby(["population", "half"]).size().unstack().to_string())
print(f"B5 context-only model agrees with a_ctx on {val_conflicts['b5_agrees_with_a_ctx'].mean():.3f} of rows")
print(val_conflicts["a_id_margin_band"].value_counts().rename("rows by a_id margin band").to_string())

half        discovery  estimation
population                       
P-player         1215         529
index-0          4296        4651
B5 context-only model agrees with a_ctx on 0.944 of rows
a_id_margin_band
> 0.05          7177
0.02 to 0.05    2303
< 0.02          1211


In [16]:
val_event_rows = val_view["event_rows"]
pool = pd.DataFrame({"event_row": val_event_rows, "window": val_view["window_index"], "position": val_view["position"]})
pool = pool[~pool["event_row"].isin(set(conflict_set["event_row"]))]
pool["match_id"] = match_ids_all[pool["event_row"]]
pool["player"] = features["player"][pool["event_row"]]
pool["a_ctx"] = context_top.loc[pool["event_row"]].to_numpy()
pool["coupled_with_prev"] = events["coupled_with_prev"].to_numpy()[pool["event_row"]]
pool["team_change"] = events["team_change"].to_numpy()[pool["event_row"]]
pool["third"] = np.minimum((events["x_norm"].to_numpy()[pool["event_row"]] * 3).astype(int), 2)
pool = pool[pool["player"] > 0]
tier_keys = {1: ["player", "a_ctx", "coupled_with_prev", "team_change", "third"],
             2: ["player", "a_ctx", "coupled_with_prev"]}
pool_groups = {tier: {key: group for key, group in pool.groupby(keys)} for tier, keys in tier_keys.items()}

p_player = val_conflicts[val_conflicts.population == "P-player"]
used = set()
control_rows = []
unmatched = 0
for _, row in p_player.iterrows():
    third = min(int(events["x_norm"].iat[int(row["event_row"])] * 3), 2)
    values = {"player": row["player"], "a_ctx": row["a_ctx"], "coupled_with_prev": row["coupled_with_prev"],
              "team_change": row["team_change"], "third": third}
    best = None
    for tier, keys in tier_keys.items():
        candidates = pool_groups[tier].get(tuple(values[k] for k in keys))
        if candidates is None:
            continue
        candidates = candidates[(candidates["match_id"] == row["match_id"]) & ~candidates["event_row"].isin(used)]
        if len(candidates):
            best = candidates.iloc[np.argmin(np.abs(candidates["position"].to_numpy() - row["position"]))]
            break
    if best is None:
        unmatched += 1
        continue
    used.add(int(best["event_row"]))
    control_rows.append({"event_row": int(best["event_row"]), "window": int(best["window"]), "position": int(best["position"]),
                         "match_id": int(match_ids_all[int(best["event_row"])]), "player": int(best["player"]),
                         "a_ctx": int(row["a_ctx"]), "a_id": int(row["a_id"]), "partner_event_row": int(row["event_row"]),
                         "partner_match_id": int(row["match_id"]), "half": row["half"], "match_tier": tier,
                         "action_id": int(targets[int(best["event_row"])]),
                         "coupled_with_prev": int(best["coupled_with_prev"])})
controls = pd.DataFrame(control_rows)
print(f"P-player rows: {len(p_player):,} | matched controls: {len(controls):,} | without a match: {unmatched:,}")
print(controls["match_tier"].value_counts().rename("controls by matching tier").to_string())
assert (controls.match_id == controls.partner_match_id).all(), "Control matching crossed a match"


P-player rows: 1,744 | matched controls: 252 | without a match: 1,492
match_tier
1    161
2     91


In [17]:
train_mask = events["split"].to_numpy() == "train"
partners = interventions.build_partner_table(events, features, train_mask)
print(partners["rule"].value_counts().to_string())

rng = np.random.default_rng(0)
donor_rows = []
p_player = val_conflicts[val_conflicts.population == "P-player"]
for _, row in p_player.iterrows():
    options = p_player[(p_player["role"] == row["role"]) & (p_player["position"] == row["position"])
                       & (p_player["match_id"] != row["match_id"]) & (p_player["half"] == row["half"])]
    rule = "same role and position, other match"
    if len(options) == 0:
        options = p_player[(p_player["position"] == row["position"]) & (p_player["match_id"] != row["match_id"]) & (p_player["half"] == row["half"])]
        rule = "same position, other match (no same-role donor)"
    if len(options) == 0:
        continue
    donor = options.iloc[int(rng.integers(0, len(options)))]
    donor_rows.append({"event_row": int(row["event_row"]), "donor_event_row": int(donor["event_row"]),
                       "donor_window": int(donor["window"]), "rule": rule})
donors = pd.DataFrame(donor_rows)
print(donors["rule"].value_counts().to_string())

rule
same role and competition       291
same role, other competition     56
no partner in role                3


rule
same role and position, other match                1584
same position, other match (no same-role donor)     159


In [18]:
n_reference = 2000 if RUN_MODE == "full" else 200
reference_windows = np.sort(np.random.default_rng(0).choice(len(val_view["windows"]), size=n_reference, replace=False))
ablation_sites = []
for layer in range(len(MODEL.blocks)):
    ablation_sites += [f"head_z_{layer}", f"mlp_out_{layer}", f"attn_out_{layer}"]
reference_means = mi_setup.reference_means(MODEL, bundle, val_view, reference_windows, ablation_sites,
                                           SETTINGS["batch_windows"], hooks.forward)
population_dir = OUT / "artifacts" / "populations"
# REPAIR F26: every population file is write-once; a re-run compares instead of overwriting.
# reference_means uses the pre-registered floating-point tolerance (1e-5), because GPU sums can differ in the last bits.
keep_or_write(population_dir / "reference_means.pt",
              lambda p: torch.save({"means": {k: v.cpu() for k, v in reference_means.items()}, "windows": reference_windows,
                                    "seed": PRIMARY_SEED}, p), tolerance=1e-5)
keep_or_write(population_dir / "val_conflicts.parquet", lambda p: val_conflicts.to_parquet(p, index=False))
keep_or_write(population_dir / "matched_controls.parquet", lambda p: controls.to_parquet(p, index=False))
keep_or_write(population_dir / "swap_partners.parquet", lambda p: partners.to_parquet(p, index=False))
keep_or_write(population_dir / "context_donors.parquet", lambda p: donors.to_parquet(p, index=False))
print("Saved:", sorted(path.name for path in population_dir.iterdir()))

Saved: ['context_donors.parquet', 'matched_controls.parquet', 'reference_means.pt', 'swap_partners.parquet', 'val_conflicts.parquet']


Historical result prose removed for the new model; use the freshly executed outputs above.


In [19]:
preregistration = {
    "model": {"primary_seed": PRIMARY_SEED, "replicate_seeds": [s for s in SEEDS if s != PRIMARY_SEED],
              "run_dirs": {str(s): str(mi_setup.run_dir_for_seed(runs, s)) for s in SEEDS}},
    "conflict_set_sha256": json.load(open(MANIFEST))["conflict_set_content"]["sha256"],
    "populations": {
        "P-all (val)": fingerprint.rowset_sha256(val_conflicts["event_id"]),
        "P-player (val)": fingerprint.rowset_sha256(val_conflicts.loc[val_conflicts.population == "P-player", "event_id"]),
        "matched controls": fingerprint.rowset_sha256(controls["event_row"]) if len(controls) else None,
    },
    "discovery_matches": discovery_matches,
    "analysis_id": 'benchmark_4L_2H_K16_wd0.1_2026-10-05',
    "status": "EXPLORATORY rerun plan; historical test outcomes already seen",
    "arbitration_metric": "Delta = logit(a_ctx) - logit(a_id), benchmark wd0.1 raw logits",
    "confirmatory_tests": {
        "C1": "mean Delta on P-all > 0; decided by the lower end of the 95% match-bootstrap CI (estimation half; test in MB12)",
        "C2": "on P-player, |change in Delta| under a matched player swap exceeds that under a norm-matched random vector; one-sided paired match-bootstrap CI above 0",
        "C3": "the model's swap-induced change in centred log-probabilities correlates positively with the player-only reference's change; CI of mean r above 0",
        "C4": "the identity x context interaction exceeds the direct-path (LayerNorm-only) floor; paired CI above 0",
    },
    "provisional_numbers": {"floating_point_tolerance": 1e-5, "bh_q": 0.05, "bh_q_sensitivity": [0.01, 0.10],
                            "recovery": "pooled over events, sign-aligned: sum(sign(d) * (m_patched - m_corrupted)) / sum(|d|), d = m_clean - m_corrupted; no per-event cut-off",
                            "random_draws": SETTINGS["n_draws"], "display_min_pair_count": 30},
    "candidate_rule_MB06": "discovery half; denoising change in Delta aligned by the sign of each event's clean-minus-corrupted difference; match-bootstrap p-value; Benjamini-Hochberg q = 0.05 within each contrast x direction; a site-position passing for the primary identity contrast (player removal) or the context contrast is a candidate",
    "fallback_rule_MB04": "NEW BENCHMARK PLAN: retain player regardless of C2; report failed C2 without outcome-dependent identity switching",
    "settings": SETTINGS, "run_mode": RUN_MODE, "written_at": fingerprint.now(),
}
config_dir = OUT / "config"
# REPAIR F26: write-once; on a re-run the existing preregistration (and its original date) is kept.
keep_or_write(config_dir / "preregistration.json", lambda p: mi_setup.save_json(p, preregistration))
mi_config = {"identity_variable": "player", "primary_seed": PRIMARY_SEED, "seeds": SEEDS}
if not (config_dir / "mi_config.json").exists():
    mi_setup.save_json(config_dir / "mi_config.json", mi_config)
fingerprint.record_files(MANIFEST, {
    "preregistration.json": config_dir / "preregistration.json",
    "populations/val_conflicts.parquet": population_dir / "val_conflicts.parquet",
    "populations/matched_controls.parquet": population_dir / "matched_controls.parquet",
    "populations/swap_partners.parquet": population_dir / "swap_partners.parquet",
    "populations/context_donors.parquet": population_dir / "context_donors.parquet",
    "populations/reference_means.pt": population_dir / "reference_means.pt",
})
# REPAIR F26: MB01 owns manifests/track_a_snapshot.json. It used to be overwritten here (with a timestamp, so its
# fingerprint changed on every run and every later notebook stopped). Now it is only compared.
track_a_changed = fingerprint.compare_snapshot(OUT / "manifests" / "track_a_snapshot.json")
if track_a_changed:
    raise RuntimeError(f"BLOCKED: Track A files changed since MB01: {track_a_changed}")
fingerprint.register(REGISTRY, {"notebook": "MB02", "experiment": "E1 verification + pre-registration",
                                "primary_seed": PRIMARY_SEED, "tests_passed": bool(test_table["passed"].all()),
                                "run_mode": RUN_MODE, "rerun": PRIOR_RUN,
                                "rerun_differences": [d[0] for d in RERUN_DIFFERENCES]})
print(json.dumps({k: preregistration[k] for k in ["model", "populations", "confirmatory_tests"]}, indent=2))

# REPAIR F26: what happened to each output, and a stop if a re-run produced anything different.
print("\nOutputs:")
for name, what in OUTPUT_LOG:
    print(f"  {name:50s} {what}")
print("Track A files changed since MB01: none")
if RERUN_DIFFERENCES:
    raise RuntimeError("BLOCKED: this re-run produced different content for "
                       + "; ".join(f"{name} ({detail})" for name, _, detail in RERUN_DIFFERENCES)
                       + ". The original files were kept; the new versions sit next to them as *.rerun_* files. "
                         "Do not continue; send this output.")
fingerprint.record_files(MANIFEST, {"mi_config.json": config_dir / "mi_config.json", "reference_models.json": config_dir / "reference_models.json", "analysis_plan.json": config_dir / "analysis_plan.json"})


{
  "model": {
    "primary_seed": 103,
    "replicate_seeds": [
      100,
      101,
      102,
      104
    ],
    "run_dirs": {
      "100": "/home/aayush/code/soccer_mi/model_training_evaluation/benchmarks/track_a_4layer/runs/transformer_T300_pre_action_L4_H2_D128_F512_K16_g-all_lr0.0003_wd0.1_s100",
      "101": "/home/aayush/code/soccer_mi/model_training_evaluation/benchmarks/track_a_4layer/runs/transformer_T300_pre_action_L4_H2_D128_F512_K16_g-all_lr0.0003_wd0.1_s101",
      "102": "/home/aayush/code/soccer_mi/model_training_evaluation/benchmarks/track_a_4layer/runs/transformer_T300_pre_action_L4_H2_D128_F512_K16_g-all_lr0.0003_wd0.1_s102",
      "103": "/home/aayush/code/soccer_mi/model_training_evaluation/benchmarks/track_a_4layer/runs/transformer_T300_pre_action_L4_H2_D128_F512_K16_g-all_lr0.0003_wd0.1_s103",
      "104": "/home/aayush/code/soccer_mi/model_training_evaluation/benchmarks/track_a_4layer/runs/transformer_T300_pre_action_L4_H2_D128_F512_K16_g-all_lr0.0003_wd0.1

{'artifacts/conflict/conflict_set.parquet': {'path': 'artifacts/conflict/conflict_set.parquet',
  'bytes': 1735548,
  'sha256': '19d6197298f38803450ba3befbc0e1b744b6ab900eecb0f6374783c4cf2263b3',
  'recorded_at': '2026-10-05T17:22:56.743259+00:00'},
 'artifacts/conflict/identity_audit/id_mlp_128_64_seed0.pt': {'path': 'artifacts/conflict/identity_audit/id_mlp_128_64_seed0.pt',
  'bytes': 330757,
  'sha256': '590fbc131586f5297fcd9ddfcdf7f2e1c503cf13e9f584142a5c31a10a79339c',
  'recorded_at': '2026-10-05T17:22:56.743711+00:00'},
 'artifacts/conflict/identity_audit/id_mlp_512_512_256_seed0.pt': {'path': 'artifacts/conflict/identity_audit/id_mlp_512_512_256_seed0.pt',
  'bytes': 2280753,
  'sha256': 'eab81928ab7743ddbe2cbdb1dc60d5b584408386e896705115145e7c8a6d62a2',
  'recorded_at': '2026-10-05T17:22:56.746435+00:00'},
 'artifacts/conflict/predictions/test_probs.parquet': {'path': 'artifacts/conflict/predictions/test_probs.parquet',
  'bytes': 7982855,
  'sha256': 'd21c146acd76ee17cb82670b

**Result (the four main tests, as written in the file).**

* **C1** — mean Δ on P-all is above 0. Decided by the lower end of the 95% match-bootstrap interval (estimation half; on test in MB12).
* **C2** — on P-player, the size of the change in Δ when the player is swapped for a matched player is larger than when the player vector is replaced by a random vector of the same length. One-sided paired match-bootstrap interval above 0.
* **C3** — the change in the model's (centred) log-probabilities caused by a player swap goes in the same direction as the change in the player-only reference. Interval of the mean correlation above 0.
* **C4** — the identity × context interaction is larger than the floor set by the direct path (LayerNorm only). Paired interval above 0.

The primary seed is 103; the replicate seeds are 100, 101, 102 and 104.

## 8. Summary

* The five SEL-4L checkpoints reproduce their recorded validation outputs exactly.
* The step-by-step forward pass computes exactly the model's function (all nine tests pass).
* Every group, control, split and main test is now fixed and fingerprinted.

MB03 reads these files and may not change them.